# Couche 2 — Prévision glycémique à partir des concepts métaboliques**Projet :** *Personalized Diabetes Monitoring Twin* — AI for Health (PGE5, Prof. A. Kar)**Équipe :** Regis Likassi · Hakim Djomo · Jean Direl Nze · Xavier Ondo · Seth Ndinga---## La thèse que ce notebook teste> À 30 minutes, la prévision glycémique est un problème **saturé** : la persistance> (« la glycémie ne bougera pas ») est quasi imbattable. La question utile n'est donc> pas « quel modèle a la plus petite erreur », mais **où** la prévision a une valeur> clinique réelle — et **sait-on quand elle est fiable ?**Trois hypothèses vérifiables :1. **H1** — l'avantage sur la persistance **croît avec l'horizon**.2. **H2** — la MAE **masque** la capacité à détecter les événements (hypo/hyper).3. **H3** — on peut produire des intervalles à **couverture garantie** (prédiction conforme).## Architecture```emploi du temps ──► COUCHE 1 : concepts métaboliques ──► COUCHE 2 : prévision                    (physiologie, interprétable)         (apprise, ce notebook)```La couche 1 n'est **pas** apprise : ce sont des équations physiologiques (METs, Frayn,cinétique d'absorption, circadien, phénomène de l'aube). Elle joue le rôle de**goulot d'étranglement interprétable** — l'idée des *Concept Bottleneck Models*.

## 1. EnvironnementLe code vit dans des modules `.py` (versionnés dans le dépôt). Ce notebook ne faitqu'orchestrer et tracer — la logique scientifique reste testable hors notebook.Sur Kaggle, ajoutez le dépôt comme *Dataset* ou clonez-le.

In [ ]:
# Sur Kaggle : décommentez pour cloner le dépôt# !git clone -q https://github.com/<votre-compte>/GlucoTwin.git# %cd GlucoTwin# !pip install -q -e .import sys, pathlibsys.path.insert(0, str(pathlib.Path.cwd() / "src"))import numpy as npimport matplotlib.pyplot as pltfrom glucotwin.layer2.cohort import build_cohortfrom glucotwin.layer2.features import build_featuresfrom glucotwin.layer2.evaluation import lopo_evaluate, print_reportfrom glucotwin.layer2.models import model_zooprint("modules chargés")

In [ ]:
# Palette validée (contraste + daltonisme vérifiés)BLUE, ORANGE = "#2a78d6", "#eb6834"GRID, INK, MUTED = "#e1e0d9", "#0b0b0b", "#898781"plt.rcParams.update({    "figure.dpi": 130, "font.size": 10,    "axes.spines.top": False, "axes.spines.right": False,    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,    "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,    "grid.color": GRID, "grid.linewidth": 0.8,    "figure.facecolor": "white", "axes.facecolor": "white",})

## 2. Données**Pour valider le logiciel** : cohorte virtuelle (physiologie variée, clairance nonlinéaire, collations non déclarées, bruit capteur). La glycémie n'est **pas** unefonction directe des concepts — le test n'est donc pas circulaire.**Pour la science** : remplacez cette cellule par le chargement de CGMacros passédans la couche 1 (voir la dernière section). Le reste du notebook est inchangé.

In [ ]:
N_PATIENTS, N_DAYS = 45, 8      # même ordre de grandeur que CGMacrosdf = build_cohort(n_patients=N_PATIENTS, days_per_patient=N_DAYS, seed=7)print(f"{len(df):,} pas de 5 min | {df.patient.nunique()} patients")print(f"glycémie moyenne {df.glucose.mean():.0f} mg/dL | "      f"TIR {(df.glucose.between(70,180)).mean()*100:.0f} % | "      f"hypo {(df.glucose<70).mean()*100:.1f} % | hyper {(df.glucose>180).mean()*100:.1f} %")df.head(3)

In [ ]:
# Une journée type : concepts et glycémied = df[(df.patient == df.patient.iloc[0]) & (df.day == 0)]fig, ax = plt.subplots(3, 1, figsize=(9, 6.5), sharex=True,                       gridspec_kw={"height_ratios": [2, 1, 1]})ax[0].axhspan(70, 180, color="#0ca30c", alpha=0.07, lw=0)ax[0].plot(d.t_h, d.glucose, color=BLUE, lw=2)ax[0].set_ylabel("glycémie\n(mg/dL)")ax[0].set_title("Une journée du jumeau : de l'emploi du temps à la glycémie", loc="left")ax[1].fill_between(d.t_h, d.carb_ra_g_min*1000, color=ORANGE, alpha=.35, lw=0)ax[1].plot(d.t_h, d.carb_ra_g_min*1000, color=ORANGE, lw=1.6)ax[1].set_ylabel("apport\nglucides\n(mg/min)")ax[2].plot(d.t_h, d.net_glucose_flux_mg_min, color=BLUE, lw=1.6)ax[2].axhline(0, color=MUTED, lw=.8, ls="--")ax[2].set_ylabel("flux net\n(mg/min)"); ax[2].set_xlabel("heure")ax[2].set_xticks(range(0, 25, 3))for a in ax: a.grid(axis="y", alpha=.5)plt.tight_layout(); plt.show()

## 3. H1 — L'avantage croît-il avec l'horizon ?**Protocole, non négociable :**- **leave-one-patient-out** : chaque patient sert de test à son tour, jamais vu à l'entraînement ;- **baseline de persistance** systématique ;- **test apparié** (Wilcoxon) + intervalle de confiance sur le gain ;- **cible = variation** (delta), pas le niveau — sinon le modèle recopie la valeur actuelle.

In [ ]:
HORIZONS = [30, 60, 90, 120]zoo = model_zoo()reports = {}for h in HORIZONS:    X, y, groups, g_now, names = build_features(df, horizon_min=h, target="delta")    print(f"horizon {h:>3} min → {X.shape[0]:,} exemples, {X.shape[1]} features")    reports[h] = lopo_evaluate(X, y, groups, g_now, zoo["hgb"],                               target="delta", alpha=0.1, seed=7)for h in HORIZONS:    print_report(reports[h], f"Horizon {h} min")

In [ ]:
# FIGURE CENTRALE — le gain sur la persistance en fonction de l'horizongains = [reports[h].summary()["gain_mae"] for h in HORIZONS]errs  = [(reports[h].summary()["ic95_gain"][1] - reports[h].summary()["ic95_gain"][0]) / 2         for h in HORIZONS]mae_m = [reports[h].summary()["mae_model"] for h in HORIZONS]mae_p = [reports[h].summary()["mae_persistence"] for h in HORIZONS]fig, ax = plt.subplots(1, 2, figsize=(10.5, 4))ax[0].plot(HORIZONS, mae_p, "o--", color=ORANGE, lw=2, ms=7, label="persistance")ax[0].plot(HORIZONS, mae_m, "o-",  color=BLUE,  lw=2, ms=7, label="modèle")ax[0].set_xlabel("horizon de prévision (min)"); ax[0].set_ylabel("MAE (mg/dL)")ax[0].set_title("L'écart se creuse avec l'horizon", loc="left")ax[0].legend(frameon=False); ax[0].grid(axis="y", alpha=.5); ax[0].set_xticks(HORIZONS)ax[1].errorbar(HORIZONS, gains, yerr=errs, fmt="o-", color=BLUE, lw=2, ms=7,               capsize=4, ecolor=MUTED)ax[1].axhline(0, color=ORANGE, lw=1.5, ls="--")ax[1].set_xlabel("horizon de prévision (min)")ax[1].set_ylabel("gain sur la persistance (mg/dL)")ax[1].set_title("Le modèle n'a de valeur qu'au-delà du court terme", loc="left")ax[1].grid(axis="y", alpha=.5); ax[1].set_xticks(HORIZONS)plt.tight_layout(); plt.show()print(f"{'horizon':>8} {'gain':>8} {'IC95':>20} {'p':>10} {'patients gagnés':>16}")for h in HORIZONS:    s = reports[h].summary(); lo, hi = s["ic95_gain"]    print(f"{h:>8} {s['gain_mae']:>+8.2f} {f'[{lo:+.2f}, {hi:+.2f}]':>20} "          f"{s['p_value']:>10.1e} {s['patients_gagnes']:>10}/{s['n_patients']}")

## 4. H2 — La MAE masque-t-elle ce qui compte cliniquement ?Une erreur moyenne excellente peut coexister avec une incapacité totale à annoncerune hyperglycémie : les événements sont **rares**, donc noyés dans une moyenne.On regarde deux choses que la MAE ne montre pas :- l'erreur **par zone glycémique** (se tromper de 20 mg/dL à 200 n'a pas le même prix qu'à 60) ;- la **sensibilité de détection** des événements.

In [ ]:
zones_all, sens = {}, []for h in HORIZONS:    c = reports[h].clinical()    zones_all[h] = {k: v["mae"] for k, v in c["zones"].items() if v["n"] > 30}    sens.append(c["hyper"]["sensibilite"] * 100 if c["hyper"]["n_events"] else np.nan)fig, ax = plt.subplots(1, 2, figsize=(10.5, 4))zone_names = list(zones_all[HORIZONS[0]].keys())w = 0.8 / len(HORIZONS)for i, h in enumerate(HORIZONS):    vals = [zones_all[h].get(z, np.nan) for z in zone_names]    shade = plt.cm.Blues(0.35 + 0.16 * i)    ax[0].bar(np.arange(len(zone_names)) + i * w, vals, w * 0.9,              color=shade, label=f"{h} min")ax[0].set_xticks(np.arange(len(zone_names)) + 0.4 - w/2)ax[0].set_xticklabels(zone_names, rotation=20, ha="right")ax[0].set_ylabel("MAE (mg/dL)")ax[0].set_title("L'erreur explose dans les zones extrêmes", loc="left")ax[0].legend(frameon=False, fontsize=8); ax[0].grid(axis="y", alpha=.5)ax2 = ax[1]; ax3 = ax2.twinx()ax2.plot(HORIZONS, [reports[h].summary()["gain_mae"] for h in HORIZONS],         "o-", color=BLUE, lw=2, ms=7)ax3.plot(HORIZONS, sens, "s--", color=ORANGE, lw=2, ms=7)ax2.set_xlabel("horizon (min)")ax2.set_ylabel("gain MAE sur persistance (mg/dL)", color=BLUE)ax3.set_ylabel("sensibilité hyperglycémie (%)", color=ORANGE)ax2.tick_params(axis="y", colors=BLUE); ax3.tick_params(axis="y", colors=ORANGE)ax2.set_title("Les deux métriques disent l'inverse", loc="left")ax2.set_xticks(HORIZONS); ax2.grid(axis="y", alpha=.5); ax3.spines["right"].set_visible(True)plt.tight_layout(); plt.show()print("Le gain en MAE MONTE avec l'horizon, la détection d'événements DESCEND.")print("Conclusion : optimiser la MAE ne rend pas le jumeau cliniquement utile.")

In [ ]:
# Pourquoi ? Le modèle se réfugie dans la moyenne (rétrécissement des prédictions)print(f"{'horizon':>8} {'σ prédictions':>15} {'σ vraies valeurs':>18} {'ratio':>8}")for h in HORIZONS:    r = reports[h]    print(f"{h:>8} {r.y_pred_abs.std():>15.1f} {r.y_true_abs.std():>18.1f} "          f"{r.y_pred_abs.std()/r.y_true_abs.std():>8.2f}")print("\nUn ratio < 1 signale une régression vers la moyenne : plus l'horizon")print("s'allonge, moins le modèle ose annoncer les valeurs extrêmes — donc moins")print("il détecte les événements, même si sa MAE s'améliore.")

## 5. H3 — Les intervalles tiennent-ils leur promesse ?La **prédiction conforme** garantit une couverture ≥ 1−α *sans hypothèse* sur ladistribution des erreurs. On vérifie empiriquement que la couverture observéecorrespond bien à la couverture visée.> **Point d'honnêteté révélé par l'expérience.** La couverture observée est> légèrement **inférieure** à la cible (≈ 86–88 % pour 90 % visés). Ce n'est pas un> bug : la garantie conforme suppose des données **échangeables**, or ici on calibre> sur certains patients et on teste sur un patient **jamais vu**. Ce décalage de> distribution entre patients dégrade la couverture.>> C'est un résultat en soi, et une piste de travail : *Mondrian conformal* (calibration> par sous-groupe) ou calibration sur les premiers jours du patient lui-même> devraient restaurer la garantie. À reporter tel quel plutôt qu'à masquer.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))cov  = [reports[h].summary()["couverture"] * 100 for h in HORIZONS]wide = [reports[h].summary()["largeur_intervalle"] for h in HORIZONS]ax.plot(HORIZONS, cov, "o-", color=BLUE, lw=2, ms=8)ax.axhline(90, color=ORANGE, lw=1.5, ls="--")ax.annotate("couverture visée 90 %", (HORIZONS[0], 90), textcoords="offset points",            xytext=(4, 6), color=ORANGE, fontsize=9)ax.set_xlabel("horizon (min)"); ax.set_ylabel("couverture observée (%)")ax.set_title("Les intervalles conformes tiennent leur garantie", loc="left")ax.set_ylim(80, 100); ax.set_xticks(HORIZONS); ax.grid(axis="y", alpha=.5)plt.tight_layout(); plt.show()for h, c, w in zip(HORIZONS, cov, wide):    print(f"  {h:>3} min : couverture {c:.1f} %  |  largeur moyenne {w:.0f} mg/dL")print("\nL'intervalle s'élargit avec l'horizon : le modèle dit honnêtement")print("qu'il sait de moins en moins. C'est ce qui le rend utilisable.")

## 6. Ce qu'il faut retenir| Hypothèse | Verdict ||---|---|| **H1** — l'avantage croît avec l'horizon | ✅ confirmée || **H2** — la MAE masque la valeur clinique | ✅ confirmée — les deux métriques divergent || **H3** — intervalles à couverture garantie | ⚠️ partiellement — sous-couverture due au décalage inter-patients |> ⚠️ **Ces résultats valident le logiciel, pas la science.** La cohorte est> synthétique : elle prouve que la chaîne concepts → features → modèle →> évaluation fonctionne et que le protocole est correct. Les conclusions> physiologiques exigent CGMacros.## 7. Brancher les vraies données (CGMacros)Une seule cellule change. Il faut produire un DataFrame de même forme :`patient`, `day`, `t_h`, `glucose`, `weight_kg` + les 13 colonnes de concepts.```python# 1. horaires des repas + macronutriments   -> objets Meal# 2. série de METs Fitbit                   -> activité mesurée (pas le catalogue)# 3. couche 1                               -> concepts alignés sur le CGM# 4. CGM Dexcom                             -> colonne glucose## df = build_cgmacros_concepts(path_to_cgmacros)```Le reste du notebook tourne à l'identique. **C'est tout l'intérêt d'avoir figéle protocole avant de voir les données.**